# ## 1. Dataset Generation & ML Baseline


In [ ]:
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support
from langchain.tools import tool

np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'transaction_id': [f"tx_{i}" for i in range(n_samples)],
    'amount': np.random.exponential(scale=1500, size=n_samples),
    'velocity_10min': np.random.poisson(lam=1.5, size=n_samples),
    'ip_distance_km': np.random.exponential(scale=30, size=n_samples),
    'failed_attempts_24h': np.random.poisson(lam=0.3, size=n_samples),
    'is_fraud': np.random.choice([0, 1], size=n_samples, p=[0.96, 0.04])
})

# Inject anomalies into fraudulent records
data.loc[data['is_fraud'] == 1, 'amount'] *= 4.5
data.loc[data['is_fraud'] == 1, 'velocity_10min'] += 6
data.loc[data['is_fraud'] == 1, 'ip_distance_km'] += 250
data.loc[data['is_fraud'] == 1, 'failed_attempts_24h'] += 3

X = data[['amount', 'velocity_10min', 'ip_distance_km', 'failed_attempts_24h']]
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Tuned Isolation Forest
model = IsolationForest(contamination=0.045, random_state=42)
model.fit(X_train.values)

print("Section 1 Complete: Model retrained with optimized parameters.\n")

Section 1 Complete: Model retrained with optimized parameters.



# ## 2. Agentic Tools & Triage Logic


In [ ]:
@tool
def run_anomaly_detector(amount: float, velocity_10min: int, ip_distance_km: float, failed_attempts_24h: int) -> str:
    """Evaluates payment payload using the trained IsolationForest model and returns anomaly score."""
    features = np.array([[amount, velocity_10min, ip_distance_km, failed_attempts_24h]])

    pred_raw = model.predict(features)[0]
    score = float(model.decision_function(features)[0])

    # Balanced risk thresholds
    if score < -0.04:
        risk_level = "HIGH"
    elif score < 0.01:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return json.dumps({
        "is_flagged_anomaly": int(pred_raw == -1),
        "anomaly_score": round(score, 4),
        "risk_level": risk_level
    })

@tool
def calculate_financial_friction(amount: float, risk_level: str) -> str:
    """Calculates false-positive cost vs potential chargeback loss to determine defense strategy."""
    estimated_fp_cost = 200.0  # ₹200 friction penalty
    potential_fraud_loss = amount
    net_exposure = potential_fraud_loss - estimated_fp_cost

    if risk_level == "HIGH":
        recommendation = "FREEZE_PAYOUT" if net_exposure > 5000 else "STEP_UP_AUTH"
    elif risk_level == "MEDIUM":
        recommendation = "STEP_UP_AUTH"
    else:
        recommendation = "APPROVE"

    return json.dumps({
        "amount": amount,
        "false_positive_friction_cost": estimated_fp_cost,
        "potential_chargeback_loss": potential_fraud_loss,
        "net_exposure": net_exposure,
        "recommended_action": recommendation
    })

@tool
def execute_defensive_action(transaction_id: str, action: str, reasoning: str) -> str:
    """Executes defensive protocol and logs an immutable audit trail entry."""
    return json.dumps({
        "status": "SUCCESS",
        "transaction_id": transaction_id,
        "executed_action": action,
        "reasoning": reasoning,
        "audit_timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    })

def triage_transaction_agent(transaction_data: dict) -> dict:
    """Agent decision pipeline executing tool-calling triage."""
    tx_id = transaction_data['transaction_id']
    amt = transaction_data['amount']
    vel = transaction_data['velocity_10min']
    ip_dist = transaction_data['ip_distance_km']
    failed_att = transaction_data['failed_attempts_24h']

    ml_result = json.loads(run_anomaly_detector.invoke({
        "amount": amt,
        "velocity_10min": vel,
        "ip_distance_km": ip_dist,
        "failed_attempts_24h": failed_att
    }))

    cost_result = json.loads(calculate_financial_friction.invoke({
        "amount": amt,
        "risk_level": ml_result['risk_level']
    }))

    action = cost_result['recommended_action']
    reasoning = (f"Transaction {tx_id} flagged as {ml_result['risk_level']} risk "
                 f"(Anomaly Score: {ml_result['anomaly_score']}). "
                 f"Potential chargeback loss ₹{amt:,.2f} vs FP cost ₹200. Action chosen: {action}.")

    return json.loads(execute_defensive_action.invoke({
        "transaction_id": tx_id,
        "action": action,
        "reasoning": reasoning
    }))

print("Section 2 Complete: Agentic tools updated.\n")

Section 2 Complete: Agentic tools updated.



# ## 3. Evaluation on Held-Out Test Set


In [ ]:
test_df = X_test.copy()
test_df['actual'] = y_test

test_results = []
for idx, row in test_df.iterrows():
    tx_payload = {
        'transaction_id': f"tx_test_{idx}",
        'amount': row['amount'],
        'velocity_10min': int(row['velocity_10min']),
        'ip_distance_km': row['ip_distance_km'],
        'failed_attempts_24h': int(row['failed_attempts_24h'])
    }

    triage_output = triage_transaction_agent(tx_payload)
    test_results.append({
        'transaction_id': tx_payload['transaction_id'],
        'amount': tx_payload['amount'],
        'actual_fraud': row['actual'],
        'executed_action': triage_output['executed_action']
    })

results_df = pd.DataFrame(test_results)

# Compute performance metrics
blocked_fraud = results_df[(results_df['actual_fraud'] == 1) & (results_df['executed_action'] != 'APPROVE')]['amount'].sum()
total_fraud_val = results_df[results_df['actual_fraud'] == 1]['amount'].sum()

# Compute traditional classification metrics based on actions (Blocked/Stepped up vs Approved)
pred_binary = np.where(results_df['executed_action'] != 'APPROVE', 1, 0)
precision, recall, f1, _ = precision_recall_fscore_support(results_df['actual_fraud'], pred_binary, average='binary')

print("=" * 55)
print("      HELD-OUT TEST SET AGENTIC PERFORMANCE SUMMARY     ")
print("=" * 55)
print(f"Precision Score            : {precision:.4f}")
print(f"Recall Score               : {recall:.4f}")
print(f"F1 Score                   : {f1:.4f}")
print("-" * 55)
print("Action Breakdown:")
print(results_df['executed_action'].value_counts().to_string())
print("-" * 55)
print(f"Total Fraud Value in Test Set : ₹{total_fraud_val:,.2f}")
print(f"Total Fraud Loss Prevented    : ₹{blocked_fraud:,.2f}")
print(f"Defense Efficiency Ratio      : {(blocked_fraud/total_fraud_val)*100:.2f}%")
print("=" * 55)

      HELD-OUT TEST SET AGENTIC PERFORMANCE SUMMARY     
Precision Score            : 0.7308
Recall Score               : 1.0000
F1 Score                   : 0.8444
-------------------------------------------------------
Action Breakdown:
executed_action
APPROVE          948
STEP_UP_AUTH      35
FREEZE_PAYOUT     17
-------------------------------------------------------
Total Fraud Value in Test Set : ₹266,568.64
Total Fraud Loss Prevented    : ₹266,568.64
Defense Efficiency Ratio      : 100.00%
